### Topologic Environment Setup ###
### Python 3.11

In [ ]:
# Stage 0: Create environment
! pip install graphviz
! pip install topologicpy --upgrade

import topologicpy

from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Graph import Graph
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Matrix import Matrix
from topologicpy.Helper import Helper
from topologicpy.Plotly import Plotly

# Check Topologic version
print(Helper.Version())

from collections import defaultdict
import pprint


### Section 0: Test obj Splitting

In [ ]:
import os
import re
from tkinter import Tk, filedialog
from topologicpy.Topology import Topology

def choose_file(title="Select OBJ file"):
    root = Tk()
    root.withdraw()
    file_path = filedialog.askopenfilename(
        title=title,
        filetypes=[("OBJ files", "*.obj")]
    )
    return file_path

def choose_folder(title="Select Output Folder"):
    root = Tk()
    root.withdraw()
    folder_path = filedialog.askdirectory(title=title)
    return folder_path

def split_obj_by_object_blocks_with_remap(input_path, output_dir):
    with open(input_path, 'r') as f:
        lines = f.readlines()

    vertex_lines = [l for l in lines if l.startswith('v ')]
    texcoord_lines = [l for l in lines if l.startswith('vt ')]
    normal_lines = [l for l in lines if l.startswith('vn ')]

    current_faces = []
    current_name = "default"
    groups = {}
    for line in lines:
        if line.startswith('o ') or line.startswith('g '):
            current_name = line.strip().split()[1]
            if current_name not in groups:
                groups[current_name] = []
        elif line.startswith('f '):
            groups.setdefault(current_name, []).append(line)

    os.makedirs(output_dir, exist_ok=True)
    output_paths = []

    for name, face_lines in groups.items():
        if not face_lines:
            continue

        used_vertex_indices = set()
        face_pattern = re.compile(r'(\d+)(?:/\d*/?\d*)?')

        for face in face_lines:
            matches = face_pattern.findall(face)
            used_vertex_indices.update(int(idx) for idx in matches)

        sorted_indices = sorted(list(used_vertex_indices))
        index_map = {old_idx: new_idx+1 for new_idx, old_idx in enumerate(sorted_indices)}

        new_vertex_lines = [vertex_lines[i-1] for i in sorted_indices]
        new_faces = []
        for face in face_lines:
            new_face = []
            parts = face.strip().split()[1:]
            for part in parts:
                parts_split = part.split('/')
                v_idx = int(parts_split[0])
                new_v = str(index_map[v_idx])
                if len(parts_split) == 1:
                    new_face.append(new_v)
                elif len(parts_split) == 2:
                    new_face.append(f"{new_v}/{parts_split[1]}")
                elif len(parts_split) == 3:
                    if parts_split[1] == '':
                        new_face.append(f"{new_v}//{parts_split[2]}")
                    else:
                        new_face.append(f"{new_v}/{parts_split[1]}/{parts_split[2]}")
            new_faces.append('f ' + ' '.join(new_face) + '\n')

        out_path = os.path.join(output_dir, f"{name}.obj")
        with open(out_path, 'w') as out:
            out.writelines(new_vertex_lines)
            out.writelines(new_faces)

        output_paths.append(out_path)

    return output_paths

# ──────────────────────────────
# USER INTERACTION
# ──────────────────────────────
input_path = choose_file("📁 Select the OBJ file to split")
if not input_path:
    raise Exception("❌ No OBJ file selected. Exiting.")

output_dir = choose_folder("📂 Choose output directory for split .obj files")
if not output_dir:
    raise Exception("❌ No output folder selected. Exiting.")

# ──────────────────────────────
# PROCESS AND SHOW RESULTS
# ──────────────────────────────
paths = split_obj_by_object_blocks_with_remap(input_path, output_dir)
models = [Topology.ByOBJPath(p) for p in paths]
Topology.Show(models)
